# SCRAM - L2 observer v3 fine-tune (QLoRA on Qwen2.5-1.5B) - local, WSL2, RTX 3050 6 GB

Trains the **v3 L2 observer** (guardian): a 7-label classifier that reads either
the operator's last 6 **proposals** or a 15 s **telemetry window** and answers
`{"label", "score", "reason"}`.

v3 = v2 with the TELEMETRY side rebuilt: 16 samples over ~15 s (v2: 6 samples,
~2.6 s), every flow, and each valve's commanded vs actual position. The proposal
examples are v2's, verbatim.

Labels: `normal`, `excursion`, `sensor_fault`, `actuator_fault`,
`provenance_violation`, `rationale_mismatch`, `setpoint_drift`.

Data: `training/observer_v3_{train,val,unseen}.jsonl` (built by
`python -m training.build_observer_dataset`), formatted by
`monitor/observer_v3_format.py`, the module that will also serve the model.
`unseen` is telemetry only: held-out fault runs, channels, valve and fault shape.

Run from the repo root (or set `SCRAM_REPO`). Outputs go to `model/scram-observer-v3-gguf/`.

## 0. Setup (once) - WSL2 + CUDA + venv

**Windows side**
1. WSL2 with Ubuntu: `wsl --install -d Ubuntu-24.04` in an admin PowerShell, reboot;
   `wsl -l -v` must show VERSION 2.
2. NVIDIA driver with CUDA on WSL: the normal Windows Game Ready / Studio driver
   (R535 or newer). Do **not** install a Linux GPU driver inside WSL. Inside WSL,
   `nvidia-smi` must list the RTX 3050.
3. Optional, more RAM for WSL: `%UserProfile%/.wslconfig` with `[wsl2]`,
   `memory=10GB`, `swap=8GB`, then `wsl --shutdown`.

**Inside WSL (Ubuntu)**
```bash
sudo apt update && sudo apt install -y python3-venv python3-dev build-essential cmake git curl libcurl4-openssl-dev
python3 -m venv ~/scram-observer-venv && source ~/scram-observer-venv/bin/activate
pip install --upgrade pip
pip install unsloth
pip install --no-deps "xformers<0.0.27" "trl<0.12.0" peft accelerate bitsandbytes
pip install jupyter nbconvert ipykernel
```
`cmake` / `build-essential` / `libcurl` are for the llama.cpp build Unsloth runs in
section 6 to produce the GGUF.

**Launch** (inside WSL, repo on the Windows drive):
```bash
source ~/scram-observer-venv/bin/activate
cd /mnt/c/Users/<you>/Downloads/SCRAM-Project
jupyter nbconvert --to notebook --execute training/scram_observer_finetune.ipynb --output scram_observer_finetune.run.ipynb --ExecutePreprocessor.timeout=-1
```
or `jupyter lab` and run the cells.

**Before launching:** close GPU-using Windows apps (browsers with hardware
acceleration, games, video players, Ollama). Expected peak VRAM at these settings
is ~4-4.5 GB of the 6 GB (4-bit base ~1.3 GB; LoRA weights + paged 8-bit Adam
~0.3 GB; activations for a 2,560-token sequence with Unsloth gradient
checkpointing and chunked cross-entropy ~2-2.5 GB), leaving ~1.5 GB headroom. The
Windows desktop alone can hold 0.5-1.5 GB, so that headroom is real but thin.

In [ ]:
import os, json, torch
from pathlib import Path
REPO = Path(os.environ.get("SCRAM_REPO", Path.cwd()))
DATA = Path(os.environ.get("SCRAM_OBSERVER_DATA_DIR", REPO / "training" / "observer_clean"))
OUT = Path(os.environ.get("SCRAM_OBSERVER_OUT", REPO / "model" / "scram-observer-candidate"))
OUT.mkdir(parents=True, exist_ok=True)
FILES = {k: DATA / f"observer_v3_{k}.jsonl" for k in ("train", "val", "unseen")}
assert all(f.exists() for f in FILES.values()), f"missing data in {DATA}: python -m training.build_clean_observer_dataset"
print(torch.cuda.get_device_name(0), round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1), "GB")

## 1. Load the base model (4-bit)
Same base as the operator. 4-bit weights are what make a 6 GB GPU possible (v2 was
trained in fp16 and peaked at ~10 GB). The longest examples (telemetry) are ~2,450
tokens with the system prompt, so MAX_SEQ is 2560; section 3 asserts nothing is
truncated.

In [ ]:
from unsloth import FastLanguageModel

MODEL = "unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit"
MAX_SEQ = 2560

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = MODEL,
    max_seq_length = MAX_SEQ,
    load_in_4bit   = True,
    dtype          = None,   # auto: bf16 on an RTX 30xx
)

## 2. Attach the LoRA adapter

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r              = 16,
    lora_alpha     = 16,
    lora_dropout   = 0,
    bias           = "none",
    target_modules = ["q_proj","k_proj","v_proj","o_proj",
                      "gate_proj","up_proj","down_proj"],
    use_gradient_checkpointing = "unsloth",
    random_state   = 3407,
)
model.print_trainable_parameters()

## 3. Load and format the data
Each line is `{"messages":[system, user, assistant]}`.

In [ ]:
from datasets import load_dataset
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(tokenizer, chat_template="qwen-2.5")

def to_text(ex):
    return {"text": tokenizer.apply_chat_template(
        ex["messages"], tokenize=False, add_generation_prompt=False)}

train_ds  = load_dataset("json", data_files=str(FILES["train"]),  split="train").map(to_text)
val_ds    = load_dataset("json", data_files=str(FILES["val"]),    split="train").map(to_text)
unseen_ds = load_dataset("json", data_files=str(FILES["unseen"]), split="train")
print(train_ds, val_ds, unseen_ds)
lens = [len(tokenizer(t)["input_ids"]) for t in train_ds["text"]]
print("token length: max", max(lens), "mean", sum(lens) // len(lens))
assert max(lens) < MAX_SEQ, "raise MAX_SEQ: an example would be truncated"
print("\n--- one formatted example ---\n")
print(train_ds[0]["text"][:1500])

## 4. Train (1 epoch, responses only)
One epoch like v2: the classes are templated, so more passes mostly memorise.
`train_on_responses_only` puts the loss on the assistant JSON only.
6 GB settings: batch 1 x grad-accum 8 (the same effective batch 8 as the Colab
version) and paged 8-bit AdamW (optimizer state can page to CPU instead of OOM).

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model         = model,
    tokenizer     = tokenizer,
    train_dataset = train_ds,
    eval_dataset  = val_ds,
    args = SFTConfig(
        dataset_text_field          = "text",
        max_seq_length              = MAX_SEQ,
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 8,
        warmup_steps                = 10,
        num_train_epochs            = 1,
        learning_rate               = 2e-4,
        fp16                        = not torch.cuda.is_bf16_supported(),
        bf16                        = torch.cuda.is_bf16_supported(),
        logging_steps               = 10,
        eval_strategy               = "steps",
        eval_steps                  = 150,
        optim                       = "paged_adamw_8bit",
        weight_decay                = 0.01,
        lr_scheduler_type           = "linear",
        seed                        = 3407,
        output_dir                  = str(REPO / "training" / "_observer_v3_outputs"),
        report_to                   = "none",
        save_strategy               = "steps",   # checkpoint so a stopped run resumes, not restarts
        save_steps                  = 50,
        save_total_limit            = 2,
    ),
)

trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|im_start|>user\n",
    response_part    = "<|im_start|>assistant\n",
)

import glob
ckpts = sorted(glob.glob(str(REPO / "training" / "_observer_v3_outputs" / "checkpoint-*")),
               key=lambda p: int(p.rsplit("-", 1)[1]))
print("resuming from", ckpts[-1] if ckpts else "scratch")
stats = trainer.train(resume_from_checkpoint=ckpts[-1] if ckpts else None)
print(stats)
print("peak VRAM reserved", round(torch.cuda.max_memory_reserved() / 2**30, 2), "GB")
model.save_pretrained(str(OUT / "lora")); tokenizer.save_pretrained(str(OUT / "lora"))

## 5. Evaluate on val AND unseen
Same report shape as v2's `eval_observer_val.json` so the numbers compare:
accuracy, valid-JSON rate, per-class recall, detection rate (attack classes
flagged as any attack), false-alarm rate (normal flagged), confusion, and the
same metrics split by input type. `EVAL_N = None` runs every example (val + unseen
~1-1.5 h on a 3050); set e.g. 300 for a quick look. Reports go to `OUT`.

In [ ]:
import random
from collections import Counter, defaultdict
FastLanguageModel.for_inference(model)

LABELS = ["normal","excursion","sensor_fault","actuator_fault",
          "provenance_violation","rationale_mismatch","setpoint_drift"]
EVAL_N = None

def ask(messages_wo_answer):
    prompt = tokenizer.apply_chat_template(messages_wo_answer, tokenize=False, add_generation_prompt=True)
    ids = tokenizer(prompt, return_tensors="pt").to("cuda")
    out = model.generate(**ids, max_new_tokens=96, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True)

def parse(text):   # same rules as monitor/observer_v2_format.parse
    s, e = text.find("{"), text.rfind("}")
    if s < 0 or e <= s: return None
    try: obj = json.loads(text[s:e+1])
    except ValueError: return None
    return obj.get("label") if isinstance(obj, dict) and obj.get("label") in LABELS else None

def metrics(rows):
    if not rows: return None
    n = len(rows); valid = sum(r["pred"] is not None for r in rows)
    gold = Counter(r["gold"] for r in rows)
    recall = {c: (sum(r["gold"] == c and r["pred"] == c for r in rows) / gold[c]) if gold[c] else None
              for c in LABELS}
    att = [r for r in rows if r["gold"] != "normal"]; nor = [r for r in rows if r["gold"] == "normal"]
    conf = defaultdict(Counter)
    for r in rows: conf[r["gold"]][r["pred"] or "invalid"] += 1
    return {"n": n, "valid_json": valid / n, "accuracy": sum(r["gold"] == r["pred"] for r in rows) / n,
            "per_class_recall": recall,
            "detection_rate": (sum(r["pred"] not in (None, "normal") for r in att) / len(att)) if att else None,
            "false_alarm_rate": (sum(r["pred"] not in (None, "normal") for r in nor) / len(nor)) if nor else None,
            "n_normal": len(nor), "confusion": {g: dict(c) for g, c in conf.items()}}

def evaluate(ds, split):
    idx = list(range(len(ds)))
    if EVAL_N:
        random.Random(0).shuffle(idx); idx = idx[:EVAL_N]
    rows = []
    for i in idx:
        msgs = ds[i]["messages"]
        raw = ask(msgs[:2])
        rows.append({"gold": json.loads(msgs[2]["content"])["label"], "pred": parse(raw), "raw": raw,
                     "kind": json.loads(msgs[1]["content"])["input"]})
    rep = {"split": split, **metrics(rows),
           "by_kind": {"proposals": metrics([r for r in rows if r["kind"] == "proposals"]),
                       "telemetry": metrics([r for r in rows if r["kind"] == "telemetry"])},
           "predictions": [{"gold": r["gold"], "pred": r["pred"], "raw": r["raw"]} for r in rows]}
    json.dump(rep, open(OUT / f"eval_{split}.json", "w"), indent=1)
    fa = rep["false_alarm_rate"]
    print(f"== {split}: n={rep['n']} acc={rep['accuracy']:.3f} valid={rep['valid_json']:.3f} "
          f"detect={rep['detection_rate']:.3f} false_alarm={fa if fa is None else round(fa, 3)}")
    for c in LABELS: print(f"   {c:20s} recall={rep['per_class_recall'][c]}")
    for k, m in rep["by_kind"].items():
        if m: print(f"   by input {k:10s} n={m['n']} acc={m['accuracy']:.3f}")
    return rep

rep_val = evaluate(val_ds, "observer_val")
rep_unseen = evaluate(unseen_ds, "observer_test_unseen")

## 6. Export GGUF for Ollama
Merges the LoRA adapter into the base and quantises to `q4_k_m` (Unsloth builds
llama.cpp the first time - needs cmake / build-essential from section 0), then
writes a Modelfile with the observer system prompt taken from a training example,
so Ollama serves exactly what was trained. `num_ctx 3072`: v3 telemetry prompts
are ~2,450 tokens.

In [ ]:
import glob, shutil
GGUF_DIR = str(OUT)
model.save_pretrained_gguf(GGUF_DIR, tokenizer, quantization_method="q4_k_m")
gguf = os.path.basename(sorted(glob.glob(GGUF_DIR + "/*.gguf"), key=os.path.getsize)[0])   # the q4_k_m one
sys_prompt = json.loads(open(FILES["train"]).readline())["messages"][0]["content"]
Q3 = chr(34) * 3
modelfile = ("FROM ./" + gguf + "\n"
             "PARAMETER temperature 0\n"
             "PARAMETER num_ctx 3072\n"
             "SYSTEM " + Q3 + sys_prompt + Q3 + "\n")
open(GGUF_DIR + "/Modelfile", "w").write(modelfile)
json.dump(stats.metrics, open(GGUF_DIR + "/train_metrics.json", "w"), indent=1)
zip_path = shutil.make_archive(str(OUT.parent / "scram-observer-v3"), "zip", GGUF_DIR)
print("wrote", zip_path, "and", GGUF_DIR)

## 7. Register with Ollama (Windows)
```powershell
cd model/scram-observer-v3-gguf
ollama create scram-observer-v3 -f Modelfile
```
Compare with v2 by input type: v2 telemetry accuracy was 71.3% val / 41.6% unseen
(sensor_fault 27.9% / 0%, actuator_fault 14.3%); the proposal side should stay
~100% / ~98%. v3 is wired in as its own step afterwards (own format module +
parity gate).